In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
# Load dataset
data_path = Path("../data/canteen_sales.csv")

df = pd.read_csv(data_path)

df.head()

,date,time_slot,day_of_week,month,item,category,price,quantity_sold,revenue,is_saturday,is_sunday,is_exam_period,is_holiday,weather,temperature,rainfall,college_event
0,2025-07-01,08:00:00,Tuesday,7,Vada Pav,Snacks,20,35,700,0,0,0,0,Cloudy,23.66,1.56,0
1,2025-07-01,08:00:00,Tuesday,7,Samosa,Snacks,15,33,495,0,0,0,0,Cloudy,23.66,1.56,0
2,2025-07-01,08:00:00,Tuesday,7,Kachori,Snacks,20,20,400,0,0,0,0,Cloudy,23.66,1.56,0
3,2025-07-01,08:00:00,Tuesday,7,Misal Pav,Snacks,50,15,750,0,0,0,0,Cloudy,23.66,1.56,0
4,2025-07-01,08:00:00,Tuesday,7,Bread Pakoda,Snacks,25,15,375,0,0,0,0,Cloudy,23.66,1.56,0


In [3]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 289525
Columns: 17


In [4]:
df.columns.tolist()

['date',
 'time_slot',
 'day_of_week',
 'month',
 'item',
 'category',
 'price',
 'quantity_sold',
 'revenue',
 'is_saturday',
 'is_sunday',
 'is_exam_period',
 'is_holiday',
 'weather',
 'temperature',
 'rainfall',
 'college_event']

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 289525 entries, 0 to 289524
Data columns (total 17 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   date            289525 non-null  object 
 1   time_slot       289525 non-null  object 
 2   day_of_week     289525 non-null  object 
 3   month           289525 non-null  int64  
 4   item            289525 non-null  object 
 5   category        289525 non-null  object 
 6   price           289525 non-null  int64  
 7   quantity_sold   289525 non-null  int64  
 8   revenue         289525 non-null  int64  
 9   is_saturday     289525 non-null  int64  
 10  is_sunday       289525 non-null  int64  
 11  is_exam_period  289525 non-null  int64  
 12  is_holiday      289525 non-null  int64  
 13  weather         289525 non-null  object 
 14  temperature     289525 non-null  float64
 15  rainfall        289525 non-null  float64
 16  college_event   289525 non-null  int64  
dtypes: float64

In [6]:
df.isnull().sum()

date              0
time_slot         0
day_of_week       0
month             0
item              0
category          0
price             0
quantity_sold     0
revenue           0
is_saturday       0
is_sunday         0
is_exam_period    0
is_holiday        0
weather           0
temperature       0
rainfall          0
college_event     0
dtype: int64

In [7]:
df["date"] = pd.to_datetime(df["date"])

df["time_slot"] = pd.to_datetime(
    df["time_slot"],
    format="%H:%M:%S"
).dt.time

In [8]:
df[["date", "time_slot"]].head()

,date,time_slot
0,2025-07-01,08:00:00
1,2025-07-01,08:00:00
2,2025-07-01,08:00:00
3,2025-07-01,08:00:00
4,2025-07-01,08:00:00


In [9]:
# =========================
# ML DATA PREPARATION
# =========================

# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Convert time_slot to time
df["time_slot"] = pd.to_datetime(
    df["time_slot"],
    format="%H:%M:%S"
).dt.time


# Check date range
print("Date range:")
print(df["date"].min(), "to", df["date"].max())


# Check number of unique food items
print("\nNumber of food items:")
print(df["item"].nunique())


# List food items
print("\nFood items:")
print(df["item"].unique())


# Check time slots
print("\nNumber of time slots:")
print(df["time_slot"].nunique())


# Check categories
print("\nCategories:")
print(df["category"].unique())

Date range:
2025-07-01 00:00:00 to 2026-06-30 00:00:00

Number of food items:
25

Food items:
['Vada Pav' 'Samosa' 'Kachori' 'Misal Pav' 'Bread Pakoda' 'Idli'
 'Medu Vada' 'Masala Dosa' 'Plain Dosa' 'Uttapam' 'Roti Sabji' 'Dal Rice'
 'Veg Thali' 'Jeera Rice' 'Paneer Masala' 'Tea' 'Coffee' 'Cold Drink'
 'Lemon Juice' 'Buttermilk' 'Veg Sandwich' 'Cheese Sandwich' 'Veg Burger'
 'Maggi' 'Veg Momos']

Number of time slots:
37

Categories:
['Snacks' 'South Indian' 'Meals' 'Beverages' 'Fast Food']


In [10]:
# =========================
# TARGET VARIABLE ANALYSIS
# =========================

print("Quantity sold statistics:")
print(df["quantity_sold"].describe())

Quantity sold statistics:
count    289525.000000
mean         21.476933
std          22.772842
min           0.000000
25%          10.000000
50%          16.000000
75%          24.000000
max         364.000000
Name: quantity_sold, dtype: float64


In [11]:
# Average demand for each item

item_demand = (
    df.groupby("item")["quantity_sold"]
      .agg(["mean", "sum", "max"])
      .sort_values("mean", ascending=False)
)

item_demand

,mean,sum,max
item,,,
Tea,52.283654,605497,364
Vada Pav,39.465158,457046,296
Coffee,35.764442,414188,249
Samosa,32.568517,377176,233
Cold Drink,27.541577,318959,188
Maggi,26.092479,302177,177
Buttermilk,24.516536,283926,150
Roti Sabji,23.176237,268404,133
Veg Thali,21.083240,244165,121


In [12]:
# =========================
# FEATURE ENGINEERING
# =========================

# Extract useful date features
df["year"] = df["date"].dt.year
df["day_of_month"] = df["date"].dt.day
df["day_of_year"] = df["date"].dt.dayofyear


# Extract time features
df["hour"] = df["time_slot"].apply(lambda x: x.hour)
df["minute"] = df["time_slot"].apply(lambda x: x.minute)


# Convert time into a simple time index
df["time_index"] = df["hour"] * 60 + df["minute"]


# Display the new features
df[
    [
        "date",
        "time_slot",
        "item",
        "quantity_sold",
        "year",
        "day_of_month",
        "day_of_year",
        "hour",
        "minute",
        "time_index"
    ]
].head()

,date,time_slot,item,quantity_sold,year,day_of_month,day_of_year,hour,minute,time_index
0,2025-07-01,08:00:00,Vada Pav,35,2025,1,182,8,0,480
1,2025-07-01,08:00:00,Samosa,33,2025,1,182,8,0,480
2,2025-07-01,08:00:00,Kachori,20,2025,1,182,8,0,480
3,2025-07-01,08:00:00,Misal Pav,15,2025,1,182,8,0,480
4,2025-07-01,08:00:00,Bread Pakoda,15,2025,1,182,8,0,480


In [13]:
# =========================
# STEP 3 — FEATURE ENGINEERING
# =========================

# Extract hour and minute from time_slot
df["hour"] = df["time_slot"].apply(lambda x: x.hour)
df["minute"] = df["time_slot"].apply(lambda x: x.minute)

# Display the new features
df[["time_slot", "hour", "minute"]].head()

,time_slot,hour,minute
0,08:00:00,8,0
1,08:00:00,8,0
2,08:00:00,8,0
3,08:00:00,8,0
4,08:00:00,8,0


In [14]:
# Final features for the ML model

features = [
    "hour",
    "minute",
    "day_of_week",
    "month",
    "item",
    "category",
    "price",
    "is_saturday",
    "is_sunday",
    "is_exam_period",
    "is_holiday",
    "weather",
    "college_event"
]

target = "quantity_sold"

X = df[features]
y = df[target]

print("Features:")
print(features)

print("\nTarget:")
print(target)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Features:
['hour', 'minute', 'day_of_week', 'month', 'item', 'category', 'price', 'is_saturday', 'is_sunday', 'is_exam_period', 'is_holiday', 'weather', 'college_event']

Target:
quantity_sold

X shape: (289525, 13)
y shape: (289525,)


In [15]:
from sklearn.model_selection import train_test_split

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (231620, 13)
Testing data: (57905, 13)


In [16]:
categorical_features = [
    "day_of_week",
    "item",
    "category",
    "weather"
]

numerical_features = [
    "hour",
    "minute",
    "month",
    "price",
    "is_saturday",
    "is_sunday",
    "is_exam_period",
    "is_holiday",
    "college_event"
]

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['day_of_week', 'item', 'category', 'weather']

Numerical features:
['hour', 'minute', 'month', 'price', 'is_saturday', 'is_sunday', 'is_exam_period', 'is_holiday', 'college_event']


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [19]:
model.fit(
    X_train,
    y_train
)

print("Model training completed successfully!")

Model training completed successfully!


In [20]:
y_pred = model.predict(X_test)

print("First 10 predictions:")
print(y_pred[:10])

print("\nFirst 10 actual values:")
print(y_test.iloc[:10].values)

First 10 predictions:
[19.57233333  1.01866667  6.8725     11.63607143 61.675      16.35566667
 17.21166667 51.41333333  3.43338095 25.24533333]

First 10 actual values:
[14  2  9  9 78 17 18 52  2 22]


In [21]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)

print("Model Performance")
print("-------------------------")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 2))

Model Performance
-------------------------
MAE : 2.79
RMSE: 4.53
R²  : 0.96
